# O6 — Modèle prédictif du retard

**Question :** peut-on anticiper un retard de livraison **au moment de la commande** ?

**Métriques définies au cadrage :**

| Métrique | Définition | Nom standard |
|---|---|---|
| **M-11** | Retards détectés ÷ total des retards réels | Rappel (*recall*) |
| **M-12** | Fausses alertes ÷ total des alertes | 1 − précision (*false discovery rate*) |

La justesse globale est écartée : avec 6,77 % de retards, dire « jamais de retard » donnerait 93,2 % de bonnes réponses sans détecter un seul cas.

**Plan du notebook**

| Partie | Contenu |
|---|---|
| 0-2 | Environnement, chargement, préparation des données |
| Q4 | Premier modèle : la régression logistique |
| Q5 | Le seuil d'alerte, et l'explication d'une anomalie |
| Q6 | Un modèle plus puissant : la forêt aléatoire |
| Q7 | Ce que le modèle a appris |
| Q8 | Où le modèle se trompe |
| Bonus | Découpage au hasard contre découpage dans le temps |

Les questions Q1 à Q3 (choix des variables, règle de référence, découpage) sont traitées en SQL : voir `SQL/O6_requetes.sql`.

**Convention :** chaque étape indique **pourquoi** on la fait, **ce que fait le code**, le **résultat obtenu** et **ce qu'on en déduit**.

## 0. Vérifier l'environnement

**Pourquoi :** s'assurer que scikit-learn, la bibliothèque de machine learning, est installée dans le bon noyau Python.

In [ ]:
import sklearn
print(sklearn.__version__)

**Résultat :** `1.8.0` → l'environnement est prêt.

## 1. Charger les données

**Pourquoi :** récupérer dans pandas les commandes avec les seules variables connues au moment de l'achat (Q1).

**Ce que fait le code :**
- se connecter à PostgreSQL avec les identifiants du fichier `.env` (même mécanisme que `distances.py`) ;
- exécuter la requête et ranger le résultat dans un tableau `df`.

`promised_days` est créée dans la requête : c'est le délai promis au client, en jours.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Connexion à la base (identifiants lus dans le .env, jamais écrits en clair)
load_dotenv()

user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
database = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
)

# Variables connues AU MOMENT DE L'ACHAT (audit Q1) + la cible is_late
requete = """
SELECT
    order_id,
    order_purchase_timestamp,
    customer_state,
    purchase_year,
    purchase_month,
    purchase_dow,
    nb_items,
    nb_sellers,
    total_price,
    total_freight,
    payment_total,
    distance_km,
    order_estimated_delivery_date::DATE
        - order_purchase_timestamp::DATE AS promised_days,
    is_late
FROM mart.orders;
"""

df = pd.read_sql(requete, engine)

**Contrôles du chargement.** On vérifie qu'aucune ligne n'a été perdue.

- `shape` : nombre de lignes et de colonnes
- `isna().sum()` : nombre de cases vides par colonne
- `mean()` de `is_late` : le taux de retard (True = 1, False = 0)

In [ ]:
print(df.shape)
print(df.isna().sum())
print(df["is_late"].mean())

**Résultat :**

| Contrôle | Attendu | Obtenu |
|---|---|---|
| Lignes × colonnes | 96 447 | ✅ 96 447 × 14 |
| `distance_km` vide | 476 | ✅ 476 |
| `payment_total` vide | 1 | ✅ 1 |
| Taux de retard | 6,77 % | ✅ 0,0677 |

**Déduction :** les données sont chargées sans perte.

## 2. Préparer les données

### 2.1 Les valeurs manquantes

**Pourquoi :** la régression logistique refuse les cases vides. Avant d'exclure les 477 commandes concernées, on vérifie que leur exclusion ne fausse rien.

**Ce que fait le code :** `df["distance_km"].isna()` repère les lignes sans distance, `df[...]` ne garde que celles-ci, puis on calcule leur taux de retard.

In [ ]:
print(df[df["distance_km"].isna()]["is_late"].mean())

**Résultat :** 9,24 %, contre 6,77 % en moyenne.

**Déduction :** plus élevé, mais sur 476 commandes, soit **44 retards sur 6 534** (0,7 %). L'exclusion n'affecte pas le modèle. L'imputation par la médiane est rejetée : elle inventerait une distance qui n'existe pas.

**Ce que fait le code :** `dropna()` supprime toutes les lignes qui ont au moins une case vide. Le `df =` est indispensable : `dropna()` crée un nouveau tableau sans modifier l'ancien.

In [ ]:
df = df.dropna()
print(df.shape)

**Résultat :** 95 970 lignes = 96 447 − 477. La commande sans paiement ne faisait pas partie des commandes sans distance.

### 2.2 Traduire l'état en nombres : le one-hot encoding

**Pourquoi :** le modèle ne calcule qu'avec des nombres. Numéroter les états (SP = 1, RJ = 2…) créerait un ordre qui n'existe pas : AL « vaudrait » trois fois SP.

**Ce que fait le code :** `get_dummies` remplace `customer_state` par **une colonne par état**, qui vaut True si la commande va vers cet état. `drop_first=True` supprime la première colonne (AC) : une commande vers AC a False partout, l'information n'est pas perdue.

⚠️ À n'exécuter qu'une fois : au second passage, `customer_state` n'existe plus.

In [ ]:
df = pd.get_dummies(df, columns=["customer_state"], drop_first=True)

print(df.shape)

df.filter(like="customer_state").head(10)

**Résultat :** 39 colonnes = 13 colonnes d'origine + 26 colonnes d'états (27 états moins AC). Sur chaque ligne, une seule colonne d'état vaut True.

Note : seul l'état du **client** (destination) est encodé. L'état du vendeur n'est pas dans les variables (piste d'amélioration).

### 2.3 Découper dans le temps (Q3)

**Pourquoi :** reproduire l'usage réel, apprendre sur le passé pour prédire le futur.

**Ce que fait le code :** `df[condition]` ne garde que les lignes où la condition est vraie, comme un `WHERE` en SQL.

In [ ]:
date_coupure = "2018-06-01"

train = df[df["order_purchase_timestamp"] < date_coupure]
test = df[df["order_purchase_timestamp"] >= date_coupure]

print(train.shape, train["is_late"].mean())
print(test.shape, test["is_late"].mean())

**Résultat :**

| | Commandes | Taux de retard |
|---|---|---|
| Entraînement (→ mai 2018) | 77 445 | 7,52 % |
| Test (juin → août 2018) | 18 525 | 3,59 % (≈ 665 retards) |

**Déduction :** le test est deux fois moins exposé au retard que l'entraînement. Il mesure la résistance du modèle à un changement de contexte.

### 2.4 Séparer les variables (X) et la cible (y)

**Pourquoi :** le modèle apprend à passer des indices (X) à la réponse (y).

**Ce que fait le code :** il retire de X trois colonnes qui ne doivent pas servir à prédire :
- `is_late` : c'est la réponse, le modèle tricherait ;
- `order_id` : un identifiant sans signification ;
- `order_purchase_timestamp` : a servi au découpage ; l'année, le mois et le jour sont déjà dans d'autres colonnes.

In [ ]:
colonnes_a_retirer = ["is_late", "order_id", "order_purchase_timestamp"]

X_train = train.drop(columns=colonnes_a_retirer)
y_train = train["is_late"]

X_test = test.drop(columns=colonnes_a_retirer)
y_test = test["is_late"]

print(X_train.shape, X_test.shape)

**Résultat :** 36 variables explicatives (39 − 3).

### 2.5 Mettre les variables à la même échelle

**Pourquoi :** les variables ont des ordres de grandeur très différents (prix en milliers, distance jusqu'à 8 678 km, états à 0 ou 1). La régression logistique apprend mal dans ces conditions. La standardisation exprime chaque variable **sans unité**, en « nombre d'écarts à la moyenne ». Bonus : les poids du modèle deviennent comparables entre eux (Q7).

**Ce que fait le code :**
- `fit_transform` sur l'entraînement : **calcule** moyenne et écart-type, puis transforme ;
- `transform` seulement sur le test : **réutilise** les valeurs de l'entraînement.

Calculer la moyenne sur le test utiliserait des informations du futur : ce serait une fuite.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

## Q4. Premier modèle : la régression logistique

**Pourquoi ce modèle :** c'est le plus simple et le plus lisible. Il calcule un score en additionnant les variables pondérées, puis le transforme en probabilité avec la sigmoïde (voir la fiche méthode).

**Ce que fait le code :**
- `class_weight="balanced"` : donne plus de poids aux retards (7,5 % des commandes), pour que le modèle ne les ignore pas ;
- `max_iter=1000` : autorise jusqu'à 1 000 essais pour trouver les meilleurs poids ;
- `.fit()` : lance l'apprentissage.

In [ ]:
from sklearn.linear_model import LogisticRegression

modele = LogisticRegression(class_weight="balanced", max_iter=1000)
modele.fit(X_train_sc, y_train)

**Résultat :** modèle entraîné en 0,7 seconde.

**Évaluation avec le seuil par défaut.** `.predict()` alerte dès que la probabilité dépasse 0,5.

In [ ]:
from sklearn.metrics import recall_score, precision_score

y_pred = modele.predict(X_test_sc)

m11 = recall_score(y_test, y_pred)
m12 = 1 - precision_score(y_test, y_pred)

print("Alertes :", y_pred.sum())
print("M-11 (détection) :", round(m11 * 100, 1), "%")
print("M-12 (fausses alertes) :", round(m12 * 100, 1), "%")

**Résultat :** 14 338 alertes (77 % des commandes) · M-11 = 96,2 % · M-12 = 95,5 %.

**Déduction :** la détection de 96 % est un **artefact du volume d'alertes** : le modèle alerte presque partout. Avec `class_weight="balanced"`, les probabilités sont gonflées et beaucoup dépassent 0,5. Tel quel, le modèle est inutilisable : il faut choisir le seuil (Q5).

## Q5. Le seuil d'alerte

### 5.1 Comparaison honnête avec la règle par état

**Pourquoi :** la règle faisait 5 065 alertes sur ce test. Pour comparer, on donne au modèle **le même nombre d'alertes** : ses 5 065 commandes les plus risquées.

**Ce que fait le code :**
- `predict_proba(...)[:, 1]` : la probabilité de retard de chaque commande ;
- `np.sort(probas)[-5065]` : la 5 065ᵉ probabilité la plus haute, c'est-à-dire le seuil qui donne 5 065 alertes.

In [ ]:
import numpy as np

probas = modele.predict_proba(X_test_sc)[:, 1]

n_alertes = 5065
seuil = np.sort(probas)[-n_alertes]
y_pred_5065 = probas >= seuil

print("Seuil :", round(seuil, 3))
print("Alertes :", y_pred_5065.sum())
print("M-11 :", round(recall_score(y_test, y_pred_5065) * 100, 1), "%")
print("M-12 :", round((1 - precision_score(y_test, y_pred_5065)) * 100, 1), "%")

**Résultat :**

| 5 065 alertes | Retards détectés | M-11 | M-12 | Retard dans les alertes |
|---|---|---|---|---|
| Règle par état (SQL) | 189 | 28,1 % | 96,3 % | 3,7 % |
| **Modèle** (seuil 0,721) | **≈ 338** | **50,8 %** | **93,3 %** | **6,7 %** |

**Déduction :** à charge égale, le modèle attrape presque **deux fois plus de retards**. Mais 93 % de ses alertes restent fausses.

⚠️ Le seuil de 0,721 ne signifie pas « 72 % de risque » : dans les alertes, le vrai taux est de 6,7 %. Les probabilités sont gonflées (`class_weight`, contexte différent) : elles servent à **classer** les commandes, pas à lire un pourcentage.

*Note : la règle a été notée sur 18 603 commandes (SQL), le modèle sur 18 525 (après `dropna`). L'écart est négligeable.*

### 5.2 Le compromis selon le volume d'alertes

**Pourquoi :** fixer les paliers d'une procédure en escalade (surveillance, information du client, changement de transporteur).

**Ce que fait le code :** le même calcul qu'en 5.1, répété pour cinq volumes grâce à une boucle `for`.

In [ ]:
for part in [0.01, 0.05, 0.10, 0.20, 0.30]:
    n = int(len(probas) * part)
    seuil = np.sort(probas)[-n]
    alertes = probas >= seuil

    m11 = recall_score(y_test, alertes)
    m12 = 1 - precision_score(y_test, alertes)

    print(f"Top {int(part*100)} % : {n} alertes | M-11 = {m11*100:.1f} % | M-12 = {m12*100:.1f} %")

**Résultat :**

| Palier | Alertes | Retards détectés | M-11 | M-12 | Retard dans les alertes | vs hasard (3,6 %) |
|---|---|---|---|---|---|---|
| Top 1 % | 185 | ≈ 33 | 5,0 % | 82,2 % | 17,8 % | × 5,0 |
| Top 5 % | 926 | ≈ 87 | 13,1 % | 90,6 % | 9,4 % | × 2,6 |
| Top 10 % | 1 852 | ≈ 136 | 20,5 % | 92,7 % | 7,3 % | × 2,0 |
| Top 20 % | 3 705 | ≈ 218 | 32,8 % | 94,1 % | 5,9 % | × 1,6 |
| Top 30 % | 5 557 | ≈ 365 | 54,9 % | 93,4 % | 6,6 % | × 1,8 |

**Déduction :** plus on alerte, plus on détecte, mais moins l'alerte est fiable. Même au top 1 %, 82 % des alertes sont fausses : le modèle justifie une surveillance ou une information du client, pas une action coûteuse systématique.

**Anomalie :** la fiabilité **remonte** entre 20 et 30 % (5,9 % → 6,6 %). La tranche ajoutée contient 147 retards sur 1 852 commandes (7,9 %), plus que la tranche 10-20 %.

### 5.3 Explication de l'anomalie

**Pourquoi :** comprendre qui sont les commandes classées entre 20 et 30 %.

**Ce que fait le code :**
- copie les commandes de test et ajoute la probabilité de chacune ;
- `rank(..., pct=True)` : classe les commandes de la plus risquée (0) à la moins risquée (1) ;
- garde la tranche 20-30 % et compte ses commandes par mois.

In [ ]:
analyse = test.copy()
analyse["proba"] = probas
analyse["rang"] = analyse["proba"].rank(ascending=False, pct=True)

tranche = analyse[(analyse["rang"] > 0.20) & (analyse["rang"] <= 0.30)]

print("Commandes :", len(tranche))
print("Taux de retard :", round(tranche["is_late"].mean() * 100, 1), "%")
print(tranche["purchase_month"].value_counts())

**Résultat :** 1 852 commandes, 7,9 % de retard, dont **1 153 en août** (62 %), alors qu'août ne pèse qu'un tiers du test.

**Piste :** les commandes d'août 2018 semblaient avoir des délais promis très courts. On compare, par mois, le taux de retard et le délai promis médian.

In [ ]:
print(analyse.groupby("purchase_month")["is_late"].agg(["count", "mean"]))
print(analyse.groupby("purchase_month")["promised_days"].median())

**Résultat :**

| Mois 2018 | Commandes | Délai promis médian | Taux de retard |
|---|---|---|---|
| Juin | 6 068 | 28 jours | 1,2 % |
| Juillet | 6 134 | 20 jours | 3,3 % |
| Août | 6 323 | 14 jours | 6,2 % |

**Déduction :** en deux mois, la promesse est divisée par deux et le retard multiplié par cinq.

**Test de confirmation :** au sein d'août seulement, promesse courte contre promesse longue. Comparer dans un même mois neutralise l'effet du mois.

**Ce que fait le code :** garde les commandes d'août (`.copy()` évite un avertissement de pandas), crée `promesse_courte` (True si ≤ médiane d'août, 14 jours), puis compare les taux de retard.

In [ ]:
aout = analyse[analyse["purchase_month"] == 8].copy()

aout["promesse_courte"] = aout["promised_days"] <= aout["promised_days"].median()

print(aout.groupby("promesse_courte")["is_late"].agg(["count", "mean"]))

**Résultat :**

| Promesse (août 2018) | Commandes | Taux de retard |
|---|---|---|
| Longue (> 14 jours) | 3 027 | 1,5 % |
| Courte (≤ 14 jours) | 3 296 | **10,5 %** |

**Déduction :** 7 fois plus de retard pour les promesses courtes, dans le même mois. Le lien entre promesse et retard est très fort (lien, pas cause prouvée). Le modèle, entraîné sur une période de promesses longues, sous-estime ce risque : il place ces commandes entre 20 et 30 % au lieu de les mettre en tête.

## Q6. Un modèle plus puissant fait-il mieux ?

**Pourquoi :** la régression logistique additionne les effets séparément. La **forêt aléatoire** peut capter des combinaisons (« promesse courte **et** longue distance »).

**Ce que fait le code :** construit 200 arbres de décision et fait la moyenne de leurs réponses.
- `min_samples_leaf=50` : chaque réponse finale s'appuie sur au moins 50 commandes (limite le surapprentissage) ;
- `random_state=42` : résultat reproductible ;
- `n_jobs=-1` : utilise tous les processeurs.

Les arbres ne sont pas sensibles à l'échelle : on utilise `X_train`, non standardisé.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

foret = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
foret.fit(X_train, y_train)

probas_foret = foret.predict_proba(X_test)[:, 1]

**Comparaison des deux modèles** aux mêmes paliers d'alertes.

In [ ]:
for part in [0.01, 0.05, 0.10, 0.20, 0.30]:
    n = int(len(probas) * part)

    for nom, p in [("Logistique", probas), ("Forêt", probas_foret)]:
        seuil = np.sort(p)[-n]
        alertes = p >= seuil
        m11 = recall_score(y_test, alertes)
        m12 = 1 - precision_score(y_test, alertes)
        print(f"Top {int(part*100)} % | {nom:10} | M-11 = {m11*100:.1f} % | M-12 = {m12*100:.1f} %")

    print()

**Résultat :**

| Palier | M-11 Logistique | M-11 Forêt | M-12 Logistique | M-12 Forêt |
|---|---|---|---|---|
| Top 1 % | **5,0 %** | 2,6 % | **82,2 %** | 90,8 % |
| Top 5 % | **13,1 %** | 10,7 % | **90,6 %** | 92,3 % |
| Top 10 % | **20,5 %** | 18,2 % | **92,7 %** | 93,5 % |
| Top 20 % | **32,8 %** | 26,9 % | **94,1 %** | 95,2 % |
| Top 30 % | **54,9 %** | 37,1 % | **93,4 %** | 95,6 % |

**Déduction :** la régression logistique gagne à tous les paliers. Deux explications probables :
1. la forêt apprend des règles trop spécifiques au passé (ex. l'incident de février-mars 2018) ;
2. un arbre ne sait pas **extrapoler** : face à des promesses plus courtes que tout ce qu'il a vu, il ne prolonge pas la tendance, contrairement à la régression.

**Modèle retenu : la régression logistique**, plus robuste et plus lisible.

## Q7. Qu'a appris le modèle ?

**Pourquoi :** comprendre sur quoi le modèle fonde ses alertes, et confronter ses choix aux analyses O1 à O4.

**Ce que fait le code :** récupère les 36 poids (`coef_`), les associe aux noms des variables et les trie. Les variables étant standardisées, les poids sont comparables : positif = risque en hausse, négatif = risque en baisse.

In [ ]:
poids = pd.Series(modele.coef_[0], index=X_train.columns)
poids = poids.sort_values()

print(poids.round(2).to_string())

**Résultat (variables principales) :**

| Rang | Variable | Poids | Lecture |
|---|---|---|---|
| 1 | `promised_days` | −0,58 | Promesse longue → moins de risque |
| 2 | `purchase_year` | +0,48 | 2018 plus risqué (incident) |
| 3 | `distance_km` | +0,35 | À promesse égale, plus loin → plus de risque |
| 4 | `customer_state_RJ` | +0,32 | RJ plus risqué |
| 5 | `customer_state_SP` | −0,27 | SP moins risqué |
| 6 | `purchase_month` | +0,23 | Mois traité comme un nombre (limite) |
| 7 | `nb_sellers` | −0,20 | Hypothèse : compense la distance maximale retenue |
| 8 | `total_freight` | +0,14 | Frais de port élevés → plus de risque |
| … | `total_price` / `payment_total` | 0,01 / 0,02 | N'apportent rien de plus |
| … | `customer_state_AL` | +0,08 | Retard déjà expliqué par distance et promesse |

Poids des états mesurés **par rapport à AC** (colonne supprimée).

**Déductions :**
- **La promesse et la distance dominent ensemble** : ce qui compte est la marge que la promesse laisse au trajet. Seul, `promised_days` ne distinguait rien (Q1).
- **Les frais de port pèsent, pas le prix** : cohérent avec l'hypothèse O4 de produits lourds ou volumineux.
- **RJ a un problème propre** (+0,32), alors qu'AL, pire état en O1, n'a que +0,08 : son retard s'explique par sa distance et sa promesse, celui de RJ non. Confirme O1.

**Limites :** `purchase_year` ne servira pas pour l'avenir ; `purchase_month` devrait être encodé par mois ; prix, paiement et frais de port sont redondants et se partagent le poids.

## Q8. Où le modèle se trompe-t-il ?

**Pourquoi :** un M-11 global peut cacher des situations très différentes. On découpe la performance par état, au palier top 10 % (surveillance et information du client).

### 8.1 Reconstituer l'état de chaque commande

**Ce que fait le code :** `idxmax` trouve la colonne d'état qui vaut True et on retire le préfixe ; les commandes sans aucun True sont vers AC. On marque ensuite les commandes en alerte.

In [ ]:
colonnes_etats = analyse.filter(like="customer_state_")

analyse["etat"] = colonnes_etats.idxmax(axis=1).str.replace("customer_state_", "")
analyse.loc[~colonnes_etats.any(axis=1), "etat"] = "AC"

analyse["alerte"] = analyse["rang"] <= 0.10

print(analyse["etat"].value_counts().head())

**Résultat :** SP 8 602 · RJ 2 119 · MG 2 052 · PR 943 · RS 904 → cohérent.

### 8.2 Performance par état

**Ce que fait le code :** l'équivalent d'un `GROUP BY ... HAVING` : par état, compte les commandes, les retards, les alertes et les retards détectés (en retard **et** en alerte), calcule le taux de retard et M-11, puis garde les états avec au moins 10 retards (principe de RG-08).

In [ ]:
analyse["detecte"] = analyse["is_late"] & analyse["alerte"]

par_etat = analyse.groupby("etat").agg(
    commandes=("is_late", "count"),
    retards=("is_late", "sum"),
    alertes=("alerte", "sum"),
    detectes=("detecte", "sum"),
)

par_etat["taux_retard_%"] = (par_etat["retards"] / par_etat["commandes"] * 100).round(1)
par_etat["M11_%"] = (par_etat["detectes"] / par_etat["retards"] * 100).round(1)

par_etat = par_etat[par_etat["retards"] >= 10].sort_values("retards", ascending=False)

print(par_etat)

**Résultat :**

| État | Commandes | Retards | Alertes | Détectés | Taux de retard | M-11 |
|---|---|---|---|---|---|---|
| **SP** | 8 602 | **404** | **14** | **3** | 4,7 % | **0,7 %** |
| RJ | 2 119 | 67 | 733 | 52 | 3,2 % | 77,6 % |
| BA | 599 | 26 | 222 | 14 | 4,3 % | 53,8 % |
| MG | 2 052 | 20 | 17 | 0 | 1,0 % | 0,0 % |
| RS | 904 | 18 | 80 | 8 | 2,0 % | 44,4 % |
| PR | 943 | 16 | 8 | 2 | 1,7 % | 12,5 % |
| PE | 309 | 14 | 81 | 5 | 4,5 % | 35,7 % |
| ES | 346 | 14 | 71 | 6 | 4,0 % | 42,9 % |
| MA | 108 | 13 | 65 | 11 | 12,0 % | 84,6 % |
| MS | 133 | 13 | 22 | 9 | 9,8 % | 69,2 % |
| DF | 419 | 11 | 12 | 1 | 2,6 % | 9,1 % |
| PI | 81 | 11 | 30 | 4 | 13,6 % | 36,4 % |
| SC | 601 | 10 | 118 | 7 | 1,7 % | 70,0 % |

**Déduction :** SP concentre **61 % des retards du test** (404 sur 665), mais le modèle n'en détecte que 3. La géographie s'est **inversée** : sur l'historique, RJ (12,11 %) était presque trois fois pire que SP (4,50 %) ; sur l'été 2018, SP (4,7 %) dépasse RJ (3,2 %). Le modèle met 40 % de ses alertes sur RJ et ignore SP.

### 8.3 Pourquoi SP ?

**Ce que fait le code :** garde les commandes vers SP et affiche, par mois, le volume, le taux de retard et le délai promis médian.

In [ ]:
sp = analyse[analyse["etat"] == "SP"]

print(sp.groupby("purchase_month").agg(
    commandes=("is_late", "count"),
    taux_retard=("is_late", "mean"),
    promesse_mediane=("promised_days", "median"),
))

**Résultat :**

| Mois 2018 | Commandes SP | Délai promis médian | Taux de retard |
|---|---|---|---|
| Juin | 2 730 | 22 jours | 0,7 % |
| Juillet | 2 711 | 15 jours | 3,7 % |
| Août | 3 161 | **10 jours** | **9,0 %** |

**Déduction :** sur SP, la promesse est plus que divisée par deux et le retard multiplié par 12. Le raccourcissement des promesses explique à la fois la hausse des retards et l'échec du modèle, entraîné sur une période de promesses longues où RJ était l'état critique.

## Bonus. Découpage au hasard contre découpage dans le temps

**Pourquoi :** mesurer ce qu'un découpage au hasard aurait fait croire.

**Ce que fait le code :** `train_test_split` tire au sort 20 % des commandes pour le test ; `stratify` garde le même taux de retard des deux côtés ; `random_state=42` fixe le tirage. Le reste est identique (suffixe `_h` pour ne pas écraser le premier modèle).

In [ ]:
from sklearn.model_selection import train_test_split

X_tout = df.drop(columns=colonnes_a_retirer)
y_tout = df["is_late"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_tout, y_tout, test_size=0.2, stratify=y_tout, random_state=42
)

scaler_h = StandardScaler()
X_tr_sc = scaler_h.fit_transform(X_tr)
X_te_sc = scaler_h.transform(X_te)

modele_h = LogisticRegression(class_weight="balanced", max_iter=1000)
modele_h.fit(X_tr_sc, y_tr)
probas_h = modele_h.predict_proba(X_te_sc)[:, 1]

for part in [0.01, 0.05, 0.10, 0.20, 0.30]:
    n = int(len(probas_h) * part)
    seuil = np.sort(probas_h)[-n]
    alertes = probas_h >= seuil
    m11 = recall_score(y_te, alertes)
    m12 = 1 - precision_score(y_te, alertes)
    print(f"Top {int(part*100)} % | Hasard | M-11 = {m11*100:.1f} % | M-12 = {m12*100:.1f} %")

**Résultat.** M-12 n'est pas comparable directement : le test au hasard a 6,77 % de retards, le test temporel 3,59 %. On compare M-11 et le rapport au hasard.

| Palier | M-11 temps | M-11 hasard | Alertes vs hasard (temps) | Alertes vs hasard (hasard) |
|---|---|---|---|---|
| Top 1 % | 5,0 % | 3,4 % | × 5,0 | × 3,4 |
| Top 5 % | 13,1 % | 14,6 % | × 2,6 | × 2,9 |
| Top 10 % | 20,5 % | 25,2 % | × 2,0 | × 2,5 |
| Top 20 % | 32,8 % | 40,1 % | × 1,6 | × 2,0 |
| Top 30 % | 54,9 % | 54,2 % | × 1,8 | × 1,8 |

**Déduction :** le hasard surestime modérément le modèle (5 à 7 points de détection aux paliers 10-20 %), mais surtout il **masque** l'inversion RJ / SP et l'effet des promesses. Même dans ce cas favorable, au mieux 23 % des alertes sont justes : le retard naît surtout pendant l'acheminement (O2), après la commande.

## Amélioration. Une variable de marge

**Pourquoi :** les Q5, Q7 et Q8 ont montré que le retard dépend du **rapport** entre la promesse et le trajet, pas de l'un ou de l'autre séparément. La régression logistique ne fait que des additions : elle ne peut pas fabriquer seule une division. On lui donne donc cette combinaison toute faite, **en plus** des deux variables d'origine.

**La variable :** les jours promis par 100 km. La distance est ramenée à un minimum de 100 km, sinon une commande à 3 km donnerait une marge absurde (24 ÷ 0,03 = 800).

**Ce que fait le code :** `clip(lower=100)` remplace toute distance inférieure à 100 km par 100 ; `qcut` découpe les commandes en 5 groupes de même taille selon leur marge ; on compare leurs taux de retard.

In [ ]:
df["marge"] = df["promised_days"] / (df["distance_km"].clip(lower=100) / 100)

df["groupe_marge"] = pd.qcut(df["marge"], 5, labels=["1 très faible", "2", "3", "4", "5 très forte"])

print(df.groupby("groupe_marge", observed=True)["is_late"].agg(["count", "mean"]))

**Résultat :**

| Marge | Commandes | Taux de retard |
|---|---|---|
| 1 — très faible | 19 194 | **10,4 %** |
| 2 | 19 194 | 7,5 % |
| 3 | 19 194 | 7,3 % |
| 4 | 20 128 | 5,2 % |
| 5 — très forte | 18 260 | **3,3 %** |

**Déduction :** seul, le délai promis ne distinguait rien (≈ 23 contre ≈ 24 jours, Q1). Rapporté à la distance, il devient discriminant : **3 fois plus de retard** pour les marges très faibles. (Groupes 4 et 5 inégaux : beaucoup de marges identiques, que `qcut` ne peut pas séparer.)

**Ajout de la marge au modèle.** Même découpage, mêmes commandes de test, mêmes paliers : la comparaison est juste. On retire aussi `groupe_marge`, qui n'a servi qu'au contrôle. Suffixe `_m` pour ne pas écraser le premier modèle.

In [ ]:
train_m = df[df["order_purchase_timestamp"] < date_coupure]
test_m = df[df["order_purchase_timestamp"] >= date_coupure]

a_retirer = colonnes_a_retirer + ["groupe_marge"]
X_train_m = train_m.drop(columns=a_retirer)
X_test_m = test_m.drop(columns=a_retirer)

scaler_m = StandardScaler()
X_train_m_sc = scaler_m.fit_transform(X_train_m)
X_test_m_sc = scaler_m.transform(X_test_m)

modele_m = LogisticRegression(class_weight="balanced", max_iter=1000)
modele_m.fit(X_train_m_sc, train_m["is_late"])
probas_m = modele_m.predict_proba(X_test_m_sc)[:, 1]

for part in [0.01, 0.05, 0.10, 0.20, 0.30]:
    n = int(len(probas) * part)
    for nom, p in [("Sans marge", probas), ("Avec marge", probas_m)]:
        seuil = np.sort(p)[-n]
        alertes = p >= seuil
        m11 = recall_score(y_test, alertes)
        m12 = 1 - precision_score(y_test, alertes)
        print(f"Top {int(part*100)} % | {nom:10} | M-11 = {m11*100:.1f} % | M-12 = {m12*100:.1f} %")
    print()

**Résultat :**

| Palier | M-11 sans marge | M-11 avec marge | M-12 sans marge | M-12 avec marge |
|---|---|---|---|---|
| Top 1 % | 5,0 % | 4,8 % | 82,2 % | 82,7 % |
| Top 5 % | 13,1 % | 13,2 % | 90,6 % | 90,5 % |
| Top 10 % | 20,5 % | 20,5 % | 92,7 % | 92,7 % |
| **Top 20 %** | 32,8 % | **42,0 %** | 94,1 % | **92,5 %** |
| **Top 30 %** | 54,9 % | **60,2 %** | 93,4 % | **92,8 %** |

**Déduction :** le gain se concentre dans la zone 20-30 %, celle de l'anomalie : la marge remonte dans le classement les commandes à promesse courte que le premier modèle sous-estimait. Au top 20 %, environ **61 retards de plus** détectés, à nombre d'alertes égal. **Modèle final retenu : régression logistique avec marge.**

## Visualisation. Où tombent les retards ratés ?

**Pourquoi :** voir concrètement comment le seuil sépare les commandes, avec le modèle final (avec marge) au palier top 20 %.

**Code couleur :**

| Couleur | Signification |
|---|---|
| Vert | Retard détecté (en retard et alerté) |
| Rouge | Retard non détecté (en retard, sans alerte) |
| Jaune | Fausse alerte (à l'heure, mais alerté) |
| Noir | Les autres (à l'heure, sans alerte) |

**Ce que fait le code :**
- calcule le seuil qui donne 20 % d'alertes (modifier `palier` pour tester un autre niveau) ;
- classe chaque commande de test dans une des quatre catégories (`~` = non, `&` = et) ;
- à gauche : chaque commande selon sa distance (échelle log) et son délai promis ;
- à droite : chaque commande selon son rang de risque et sa probabilité, avec le seuil en pointillés ;
- les points noirs, très nombreux, sont petits et transparents ; rouges et verts sont dessinés en dernier pour rester visibles ;
- `clip(lower=1)` évite une distance nulle, impossible en échelle log.

In [ ]:
import matplotlib.pyplot as plt

palier = 0.20
n = int(len(probas_m) * palier)
seuil = np.sort(probas_m)[-n]

vis = test_m[["distance_km", "promised_days", "is_late"]].copy()
vis["proba"] = probas_m
vis["alerte"] = vis["proba"] >= seuil
vis["rang"] = vis["proba"].rank(ascending=False)

categories = [
    ("Autres", ~vis["is_late"] & ~vis["alerte"], "black", 2, 0.15),
    ("Fausses alertes", ~vis["is_late"] & vis["alerte"], "gold", 4, 0.4),
    ("Retards non détectés", vis["is_late"] & ~vis["alerte"], "red", 14, 0.9),
    ("Retards détectés", vis["is_late"] & vis["alerte"], "green", 14, 0.9),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

for nom, masque, couleur, taille, transparence in categories:
    etiquette = f"{nom} ({masque.sum()})"
    ax1.scatter(vis.loc[masque, "distance_km"].clip(lower=1), vis.loc[masque, "promised_days"],
                c=couleur, s=taille, alpha=transparence, label=etiquette)
    ax2.scatter(vis.loc[masque, "rang"], vis.loc[masque, "proba"],
                c=couleur, s=taille, alpha=transparence, label=etiquette)

ax1.set_xscale("log")
ax1.set_xlabel("Distance vendeur → client (km, échelle log)")
ax1.set_ylabel("Délai promis (jours)")
ax1.set_title("Où tombent les retards ?")
ax1.legend()

ax2.axhline(seuil, color="gray", linestyle="--")
ax2.axvline(n, color="gray", linestyle="--")
ax2.set_xlabel("Rang de risque (1 = la plus risquée)")
ax2.set_ylabel("Probabilité calculée par le modèle")
ax2.set_title(f"Le seuil d'alerte (top {int(palier*100)} %, seuil = {seuil:.2f})")
ax2.legend()

plt.tight_layout()
plt.savefig("../RESULTATS/O6_nuage_seuil.png", dpi=150)
plt.show()

**Résultat (top 20 %, seuil = 0,77) :** 279 retards détectés · 386 non détectés · 3 426 fausses alertes · 14 434 autres.

**Graphique de gauche :**
- les **verts** (détectés) sont surtout à droite : longues distances, plus de 1 000 km ;
- les **rouges** (ratés) se concentrent **en bas à gauche** : 10 à 300 km, promesses de 3 à 15 jours, le profil de SP à l'été 2018 ;
- les **jaunes** (fausses alertes) envahissent la zone des longues distances.

**Déduction :** le modèle associe encore le risque à la distance. La marge par kilomètre ne capte pas le **délai minimal incompressible** d'une livraison (validation, préparation de 2 à 3 jours, collecte) : une commande à 30 km promise en 5 jours obtient la même marge qu'une commande à 1 000 km promise en 50 jours, alors qu'elle est bien plus serrée.

**Graphique de droite :** les rouges s'étalent sur toute la courbe, pas seulement sous le seuil. Baisser légèrement le seuil récupérerait peu de retards : ils sont **mal classés**, pas juste sous la barre. Limite visuelle : les points se superposent sur une seule ligne ; un histogramme des probabilités serait plus lisible pour voir la répartition.

### Export pour Power BI

**Pourquoi :** Power BI ne lit pas les variables Python. On exporte une petite table, une ligne par palier et par modèle, pour construire le visuel « détectés / ratés » en phase 05.

In [ ]:
lignes = []
for part in [0.01, 0.05, 0.10, 0.20, 0.30]:
    n = int(len(probas) * part)
    for nom, p in [("Sans marge", probas), ("Avec marge", probas_m)]:
        alertes = p >= np.sort(p)[-n]
        detectes = int((alertes & y_test.values).sum())
        lignes.append({
            "palier": f"Top {int(part*100)} %",
            "modele": nom,
            "alertes": int(alertes.sum()),
            "retards_detectes": detectes,
            "retards_rates": int(y_test.sum()) - detectes,
            "fausses_alertes": int(alertes.sum()) - detectes,
        })

pd.DataFrame(lignes).to_csv("../RESULTATS/O6_seuils.csv", index=False)

## Synthèse

*Le retard est en partie prévisible à la commande : à charge égale, la régression logistique détecte deux fois plus de retards qu'une règle par état, et une variable de marge (jours promis par 100 km) ajoute encore 9 points de détection au top 20 %. Mais elle reste peu précise (au moins 82 % de fausses alertes) et devient obsolète dès qu'Olist change sa politique de promesse : à l'été 2018, le raccourcissement des délais promis, surtout vers SP, a fait exploser les retards là où le modèle ne regardait pas.*

Constats détaillés : `RESULTATS/O6_constats.md` · Méthode et notions : `RESULTATS/O6_fiche_methode.md`